# Detecção de duplicatas
**Fernando Amorim Pontes Neto**

Cada par aparece uma única vez; a diagonal é excluída. O critério é similaridade estritamente maior que o limiar.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

In [ ]:
e=codificar(textos,MODELOS[0]); matriz=cosine_similarity(e)
pares=pares_da_matriz(matriz,0.85,ids)
display(pd.DataFrame(pares,columns=['id_a','id_b','similaridade']))
pd.DataFrame(pares,columns=['id_a','id_b','similaridade']).to_csv(saida/'duplicatas.csv',index=False)
# A função solicitada retorna índices da lista; a versão com IDs utiliza a mesma matriz.
assert len(detectar_duplicatas(textos,limiar=0.85)) == len(pares)
fig,ax=plt.subplots(figsize=(11,9))
im=ax.imshow(matriz,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(40),ids,rotation=90,fontsize=7); ax.set_yticks(range(40),ids,fontsize=7)
fig.colorbar(im,ax=ax,label='Cosseno'); fig.tight_layout()
fig.savefig(saida/'heatmap.png',dpi=150); plt.show()
pd.DataFrame(matriz,index=ids,columns=ids).to_csv(saida/'matriz_similaridade.csv')

## Escolha do limiar
0,85 é o ponto de partida conservador solicitado, não uma garantia de duplicidade. O percentil 90 é calculado apenas nos 780 pares únicos, excluindo a diagonal. Esse percentil admite aproximadamente 10% dos pares por construção, mesmo quando não existem duplicatas; não equivale aos 15% de registros mencionados no enunciado. A inspeção abaixo compara vários limiares. Ajustar e medir no mesmo corpus é exploratório: uma validação real deve separar eventos na calibração e no teste para evitar vazamento.

In [ ]:
valores=matriz[np.triu_indices(len(ids),k=1)]
p90=float(np.percentile(valores,90))
print('Pares únicos:',len(valores),'Percentil 90:',round(p90,4))
limiares=sorted(set([0.65,0.70,0.75,0.80,0.85,0.90,0.95,p90]))
gabarito=Path('duplicatas_reais.json')
verdade=json.loads(gabarito.read_text()) if gabarito.exists() else None
avaliacoes=[]
for limiar in limiares:
    previstos=pares_da_matriz(matriz,limiar,ids)
    r={'limiar':limiar,'pares_detectados':len(previstos)}
    if verdade is not None:
        av=avaliar_pares(previstos,verdade,ids)
        r.update({k:av[k] for k in ['precision','recall','F1']})
    avaliacoes.append(r)
display(pd.DataFrame(avaliacoes).round(4))
resumo={'limiar':0.85,'percentil_90':p90,'pares':pares,'sensibilidade':avaliacoes}
if verdade is None:
    display(Markdown('**Sem gabarito exaustivo, não é possível calcular falsos positivos, falsos negativos, precisão ou recall.** Forneça duplicatas_reais.json como lista de pares de IDs. Pares ilustrativos do enunciado não bastam como gabarito.'))
else:
    av=avaliar_pares(pares,verdade,ids); resumo['avaliacao']=av
    display(pd.DataFrame([{k:av[k] for k in ['precision','recall','F1']}]))
    for tipo in ['TP','FP','FN']:
        print('\n',tipo,':',av[tipo])
        for a,b in av[tipo]:
            display(base[base.id.isin([a,b])][['id','data','texto']])
    display(Markdown(f"Com 0,85, foram encontrados **{len(av['TP'])} verdadeiros positivos**, **{len(av['FP'])} falsos positivos** e **{len(av['FN'])} falsos negativos**. Falsos positivos devem ser revistos quanto a locais e eventos distintos; falsos negativos indicam paráfrases pouco reconhecidas ou contexto adicional que reduz o cosseno. Alterar o limiar troca precisão por cobertura e não resolve sozinho a distinção entre tema e ocorrência."))
(saida/'duplicatas_resumo.json').write_text(json.dumps(resumo,ensure_ascii=False,indent=2))

## Decisão operacional
Os pares detectados são candidatos para revisão humana, nunca registros a excluir automaticamente. Data próxima, endereço compatível e descrição do mesmo evento fortalecem a evidência; categoria igual não basta. O gabarito deve ser revisado por quem conhece os casos e sua proveniência deve acompanhar a análise.